In [ ]:
%%sql -r dataframe_1
USE DATABASE AI_POLICY_NAVIGATOR;
USE SCHEMA RAG;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
# Install required packages
import subprocess
subprocess.run(["pip", "install", "snowflake-ml-python"], capture_output=True)
print("Done")

## Phase 1 — Parse Documents

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE AI_POLICY_NAVIGATOR.RAG.PARSED_DOCUMENTS AS
SELECT
    'EU_AI_ACT' AS source_document,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
    @AI_POLICY_NAVIGATOR.RAG.PDF_STAGE,
    'EU_AI_ACT.pdf',
    {'mode': 'LAYOUT'}
    ) AS parsed_content
UNION ALL
SELECT
    'US_AI_EO' AS source_document,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
    @AI_POLICY_NAVIGATOR.RAG.PDF_STAGE,
    'US_AI_Executive_Order.pdf',
    {'mode': 'LAYOUT'}
    ) AS parsed_content;

In [ ]:
%%sql -r dataframe_3
--> Verify
SELECT 
    source_document,
    LENGTH(parsed_content:content) AS content_length,
    LEFT(parsed_content:content, 500) AS preview
FROM AI_POLICY_NAVIGATOR.RAG.PARSED_DOCUMENTS;

## Phase 2 — Chunking

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS AS
SELECT
    source_document,
    chunk.index AS chunk_index,
    chunk.value::STRING AS chunk_text
FROM AI_POLICY_NAVIGATOR.RAG.PARSED_DOCUMENTS,
LATERAL FLATTEN(
    INPUT => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
    parsed_content:content::STRING,
    'markdown',
    500,
    50
    )
) AS chunk;

In [ ]:
%%sql -r dataframe_5
--> Verify
SELECT 
    source_document,
    COUNT(*) AS total_chunks,
    AVG(LENGTH(chunk_text)) AS avg_chunk_length,
    MIN(LENGTH(chunk_text)) AS min_chunk_length,
    MAX(LENGTH(chunk_text)) AS max_chunk_length
FROM AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS
GROUP BY source_document;

In [ ]:
%%sql -r dataframe_6
--> Quick check before be move to Phase 3
SELECT 
    source_document,
    chunk_index,
    chunk_text
FROM AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS
WHERE source_document = 'EU_AI_ACT'
ORDER BY chunk_index
LIMIT 5;

## Phase 3 — Cortex Search Service

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE CORTEX SEARCH SERVICE AI_POLICY_NAVIGATOR.RAG.POLICY_SEARCH_SERVICE
    ON chunk_text
    ATTRIBUTES source_document, chunk_index
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            chunk_text,
            source_document,
            chunk_index
        FROM AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS
    );

In [ ]:
%%sql -r dataframe_8
--> Quick check
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'AI_POLICY_NAVIGATOR.RAG.POLICY_SEARCH_SERVICE',
        '{
            "query": "What are prohibited AI practices?",
            "columns": ["chunk_text", "source_document", "chunk_index"],
            "limit": 3
        }'
    )
)['results'] AS search_results;

## Phase 4 — RAG Pipeline

In [ ]:
from snowflake.core import Root
from snowflake.snowpark.context import get_active_session

session = get_active_session()
root = Root(session)

search_service = (
root
.databases["AI_POLICY_NAVIGATOR"]
.schemas["RAG"]
.cortex_search_services["POLICY_SEARCH_SERVICE"]
)

def retrieve_chunks(query, source_filter=None, limit=5):
    filter_obj = {"@eq": {"SOURCE_DOCUMENT": source_filter}} if source_filter else None
    
    request_args = {
        "query": query,
        "columns": ["chunk_text", "source_document", "chunk_index"],
        "limit": limit
    }
    if filter_obj:
        request_args["filter"] = filter_obj
    
    response = search_service.search(**request_args)
    return response.results

print("Search service connected successfully")

In [ ]:
def build_prompt(query, chunks):
    context = ""
    for i, chunk in enumerate(chunks):
        source = chunk["source_document"]
        text = chunk["chunk_text"]
        context += f"[Chunk {i+1} - {source}]\n{text}\n\n"
    
    prompt = f"""You are an AI policy expert assistant. Answer the user's question based ONLY on the provided context from the EU AI Act and US AI Executive Order documents.

If the answer is not in the context, say "I could not find information about this in the provided documents."

Always mention which document your answer comes from.

Context:
{context}

Question: {query}

Answer:"""
    
    return prompt

print("Prompt builder ready")

### 4.1 — Hybrid Retrieval (Keyword + Semantic Search)

In [ ]:
def retrieve_chunks_hybrid(query, source_filter=None, limit=5):
    # Step 1: Semantic retrieval via Cortex Search
    semantic_chunks = retrieve_chunks(query, source_filter, limit)
    semantic_ids = set([c["chunk_index"] for c in semantic_chunks])
    
    # Step 2: Keyword retrieval via SQL ILIKE
    source_clause = f"AND source_document = '{source_filter}'" if source_filter else ""
    
    # Extract keywords from query (words longer than 4 chars)
    keywords = [w for w in query.split() if len(w) > 4]
    keyword_conditions = " OR ".join([
        f"LOWER(chunk_text) LIKE LOWER('%{kw}%')" 
        for kw in keywords
    ])
    
    if keyword_conditions:
        keyword_rows = session.sql(f"""
            SELECT chunk_text, source_document, chunk_index
            FROM AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS
            WHERE ({keyword_conditions})
            {source_clause}
            AND chunk_index NOT IN ({','.join([str(i) for i in semantic_ids]) if semantic_ids else '0'})
            LIMIT {limit}
        """).collect()
        
        keyword_chunks = [{"chunk_text": r["CHUNK_TEXT"],
                          "source_document": r["SOURCE_DOCUMENT"],
                          "chunk_index": r["CHUNK_INDEX"]} for r in keyword_rows]
    else:
        keyword_chunks = []
    
    # Step 3: Combine — semantic first, keyword results appended
    all_chunks = semantic_chunks + keyword_chunks
    return all_chunks

print("Hybrid retrieval ready")

In [ ]:
def retrieve_chunks_hybrid_expanded(query, source_filter=None, limit=5, max_chunks=20):
    # Step 1: Semantic retrieval
    semantic_chunks = retrieve_chunks(query, source_filter, limit)
    semantic_indices = set([int(c["chunk_index"]) for c in semantic_chunks])
    
    # Step 2: Detect header chunks and expand forward
    expanded_indices = set(semantic_indices)
    header_markers = [
        "shall be prohibited", "article 5", "chapter ii",
        "the following", "prohibited ai practices"
    ]
    
    for c in semantic_chunks:
        idx = int(c["chunk_index"])
        text = c["chunk_text"].lower()
        is_header = any(marker in text for marker in header_markers)
        forward = 15 if is_header else 2
        for offset in range(1, forward + 1):
            expanded_indices.add(idx + offset)
    
    # Step 3: Keyword matches as-is (no expansion)
    hybrid_chunks = retrieve_chunks_hybrid(query, source_filter, limit)
    keyword_indices = set([int(c["chunk_index"]) for c in hybrid_chunks 
                          if int(c["chunk_index"]) not in semantic_indices])
    
    all_indices = expanded_indices.union(keyword_indices)
    source_clause = f"AND source_document = '{source_filter}'" if source_filter else ""
    indices_str = ','.join([str(i) for i in sorted(all_indices)])
    
    rows = session.sql(f"""
        SELECT chunk_text, source_document, chunk_index
        FROM AI_POLICY_NAVIGATOR.RAG.CHUNKED_DOCUMENTS
        WHERE chunk_index IN ({indices_str})
        {source_clause}
        ORDER BY chunk_index
        LIMIT {max_chunks}
    """).collect()
    
    return [{"chunk_text": r["CHUNK_TEXT"],
             "source_document": r["SOURCE_DOCUMENT"],
             "chunk_index": r["CHUNK_INDEX"]} for r in rows]

print("Hybrid expanded retrieval ready")

In [ ]:
def rag_query(query, source_filter=None, limit=5):
    # Use hybrid expanded retrieval
    chunks = retrieve_chunks_hybrid_expanded(query, source_filter, limit, max_chunks=20)
    
    prompt = build_prompt(query, chunks)
    
    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS answer
    """).collect()
    
    answer = response[0]["ANSWER"]
    sources = list(set([c["source_document"] for c in chunks]))
    
    return {
        "question": query,
        "answer": answer,
        "sources": sources,
        "chunks_used": len(chunks)
    }

print("Updated RAG query ready")

In [ ]:
result = rag_query(
    "What are the prohibited AI practices under Article 5?",
    source_filter="EU_AI_ACT"
)
print(f"Chunks used: {result['chunks_used']}")
print(f"\nAnswer:\n{result['answer']}")

In [ ]:
def rag_query_balanced(query, limit_per_doc=5):
    eu_chunks = retrieve_chunks_hybrid_expanded(query, "EU_AI_ACT", limit_per_doc, max_chunks=10)
    eo_chunks = retrieve_chunks_hybrid_expanded(query, "US_AI_EO", limit_per_doc, max_chunks=10)
    
    all_chunks = eu_chunks + eo_chunks
    prompt = build_prompt(query, all_chunks)
    
    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{prompt.replace("'", "''")}'
        ) AS answer
    """).collect()
    
    sources = list(set([c["source_document"] for c in all_chunks]))
    return {
        "question": query,
        "answer": response[0]["ANSWER"],
        "sources": sources,
        "chunks_used": len(all_chunks)
    }

print("Updated balanced RAG ready")

## Phase 5 — Evaluation (LLM-as-a-Judge)

In [ ]:
%%sql -r dataframe_11
CREATE OR REPLACE TABLE AI_POLICY_NAVIGATOR.RAG.EVAL_DATASET (
    question_id INT,
    question TEXT,
    expected_source VARCHAR,
    question_type VARCHAR
);

INSERT INTO AI_POLICY_NAVIGATOR.RAG.EVAL_DATASET VALUES
(1, 'What are the prohibited AI practices under Article 5?', 'EU_AI_ACT', 'enumerated_list'),
(2, 'What is the purpose of the EU AI Act?', 'EU_AI_ACT', 'conceptual'),
(3, 'What does the US AI EO say about AI safety standards?', 'US_AI_EO', 'conceptual'),
(4, 'What are high-risk AI systems according to the EU AI Act?', 'EU_AI_ACT', 'definition'),
(5, 'What role does NIST play according to the US AI EO?', 'US_AI_EO', 'specific_entity'),
(6, 'How does the EU AI Act define an AI system?', 'EU_AI_ACT', 'definition'),
(7, 'What does the US AI EO say about AI and national security?', 'US_AI_EO', 'conceptual'),
(8, 'How do both documents address transparency in AI?', 'BOTH', 'cross_document'),
(9, 'What obligations do AI providers have under the EU AI Act?', 'EU_AI_ACT', 'specific_role'),
(10, 'What does the US AI EO say about protecting privacy?', 'US_AI_EO', 'conceptual');

In [ ]:
def llm_judge(question, answer, source_used):
    judge_prompt = f"""You are an expert evaluator of AI question-answering systems. 
Evaluate the following answer to a question about AI policy documents (EU AI Act and US AI Executive Order).

Score the answer on three dimensions, each from 1 to 5:

1. RELEVANCE: Does the answer directly address the question asked?
   1 = Completely off-topic, 5 = Perfectly on-topic

2. FAITHFULNESS: Is the answer grounded in the source documents without hallucination?
   1 = Mostly hallucinated, 5 = Fully grounded in documents

3. COMPLETENESS: Does the answer cover the question fully or just partially?
   1 = Very incomplete, 5 = Fully complete

Question: {question}
Sources used: {source_used}
Answer: {answer}

Respond ONLY in this exact JSON format, nothing else:
{{
    "relevance": <score>,
    "faithfulness": <score>,
    "completeness": <score>,
    "reasoning": "<one sentence explaining your scores>"
}}"""

    response = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            '{judge_prompt.replace("'", "''")}'
        ) AS evaluation
    """).collect()
    
    import json
    raw = response[0]["EVALUATION"].strip()
    clean = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(clean)

print("Judge function ready")

In [ ]:
import time

def run_batch_eval():
    questions = session.sql("""
        SELECT question_id, question, expected_source, question_type 
        FROM AI_POLICY_NAVIGATOR.RAG.EVAL_DATASET
        ORDER BY question_id
    """).collect()
    
    results = []
    
    for row in questions:
        qid = row["QUESTION_ID"]
        question = row["QUESTION"]
        expected_source = row["EXPECTED_SOURCE"]
        qtype = row["QUESTION_TYPE"]
        
        print(f"Evaluating Q{qid}: {question[:50]}...")
        
        # Choose retrieval strategy based on question type
        if qtype == "cross_document":
            result = rag_query_balanced(question, limit_per_doc=5)
        elif qtype == "enumerated_list":
            result = rag_query(question, source_filter="EU_AI_ACT")
        else:
            result = rag_query(question, limit=5)
        
        # Judge the answer
        scores = llm_judge(question, result["answer"], result["sources"])
        
        results.append({
            "question_id": qid,
            "question": question,
            "expected_source": expected_source,
            "question_type": qtype,
            "sources_used": str(result["sources"]),
            "answer": result["answer"],
            "relevance": scores["relevance"],
            "faithfulness": scores["faithfulness"],
            "completeness": scores["completeness"],
            "reasoning": scores["reasoning"]
        })
        
        print(f"  ✓ Relevance: {scores['relevance']} | Faithfulness: {scores['faithfulness']} | Completeness: {scores['completeness']}")
        time.sleep(1)  # small delay to avoid rate limits
    
    return results

print("Batch evaluator ready")

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE TABLE AI_POLICY_NAVIGATOR.RAG.EVAL_RESULTS (
    question_id INT,
    question TEXT,
    expected_source VARCHAR,
    question_type VARCHAR,
    sources_used VARCHAR,
    answer TEXT,
    relevance INT,
    faithfulness INT,
    completeness INT,
    reasoning TEXT,
    evaluated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType
import pandas as pd

df = pd.DataFrame([{
    "QUESTION_ID": int(r["question_id"]),
    "QUESTION": str(r["question"]),
    "EXPECTED_SOURCE": str(r["expected_source"]),
    "QUESTION_TYPE": str(r["question_type"]),
    "SOURCES_USED": str(r["sources_used"]),
    "ANSWER": str(r["answer"]),
    "RELEVANCE": int(r["relevance"]),
    "FAITHFULNESS": int(r["faithfulness"]),
    "COMPLETENESS": int(r["completeness"]),
    "REASONING": str(r["reasoning"]),
    "EVALUATED_AT": pd.Timestamp.now()
} for r in eval_results])

snowpark_df = session.create_dataframe(df)
snowpark_df.write.mode("append").save_as_table("AI_POLICY_NAVIGATOR.RAG.EVAL_RESULTS")

print(f"Saved {len(eval_results)} results successfully")

In [ ]:
%%sql -r dataframe_13
SELECT 
    question_id,
    question_type,
    expected_source,
    relevance,
    faithfulness,
    completeness,
    ROUND((relevance + faithfulness + completeness) / 3.0, 2) AS avg_score,
    reasoning
FROM AI_POLICY_NAVIGATOR.RAG.EVAL_RESULTS
ORDER BY question_id;